In [1]:
%pip install -q "mediapipe==0.10.14" "opencv-python>=4.9.0" "numpy>=1.24,<2.0"

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

for folder in ("src", "models", "assets"):
    Path(folder).mkdir(parents=True, exist_ok=True)

print("Project folders ready:", Path.cwd())

Project folders ready: /Users/varshh06/Desktop/CV-END-TO-END/03. ARTIFICIAL MAGIC SYTSTEM


In [ ]:
%%writefile src/config.py
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class Character:
    key: str
    name: str
    bgr: tuple
    accent: tuple

# OpenCV colours use BGR format.
CHARACTERS = {
    "1": Character("1", "DOCTOR STRANGE", (0, 155, 255), (80, 220, 255)),
    "2": Character("2", "SCARLET WITCH", (0, 0, 235), (70, 35, 255)),
    "3": Character("3", "AGATHA HARKNESS", (220, 35, 235), (245, 145, 255)),
    "4": Character("4", "LOKI", (45, 205, 55), (145, 255, 155)),
    "5": Character("5", "THOR", (255, 205, 65), (255, 250, 225)),
    "6": Character("6", "THANOS", (180, 50, 210), (255, 190, 100)),
}

PROJECT_ROOT = Path(__file__).resolve().parent.parent
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_PATH = MODEL_DIR / "gesture_recognizer.task"

MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "gesture_recognizer/gesture_recognizer/float16/1/"
    "gesture_recognizer.task"
)

MODEL_DIR.mkdir(parents=True, exist_ok=True)

Overwriting src/config.py


In [4]:
%%writefile src/hand_tracking.py
from pathlib import Path
from urllib.request import urlretrieve
import time

import cv2
import mediapipe as mp
import numpy as np

from config import MODEL_PATH, MODEL_URL

class HandTracker:
    """Pretrained MediaPipe Gesture Recognizer; no project-specific training."""

    def __init__(self, max_hands: int = 2):
        model_path = Path(MODEL_PATH)
        model_path.parent.mkdir(parents=True, exist_ok=True)
        if not model_path.exists():
            print("Downloading pretrained MediaPipe Gesture Recognizer...")
            urlretrieve(MODEL_URL, model_path)

        BaseOptions = mp.tasks.BaseOptions
        GestureRecognizer = mp.tasks.vision.GestureRecognizer
        GestureRecognizerOptions = mp.tasks.vision.GestureRecognizerOptions
        RunningMode = mp.tasks.vision.RunningMode

        options = GestureRecognizerOptions(
            base_options=BaseOptions(model_asset_path=str(model_path)),
            running_mode=RunningMode.VIDEO,
            num_hands=max_hands,
            min_hand_detection_confidence=0.55,
            min_hand_presence_confidence=0.50,
            min_tracking_confidence=0.50,
        )
        self.recognizer = GestureRecognizer.create_from_options(options)
        self.last_timestamp_ms = 0

    def process(self, bgr_frame):
        rgb = cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        timestamp_ms = int(time.monotonic() * 1000)
        timestamp_ms = max(timestamp_ms, self.last_timestamp_ms + 1)
        self.last_timestamp_ms = timestamp_ms

        result = self.recognizer.recognize_for_video(mp_image, timestamp_ms)
        hands = []
        for index, landmarks in enumerate(result.hand_landmarks):
            points = np.asarray(
                [[p.x, p.y, p.z] for p in landmarks], dtype=np.float32
            )
            gesture = "None"
            score = 0.0
            if index < len(result.gestures) and result.gestures[index]:
                gesture = result.gestures[index][0].category_name
                score = float(result.gestures[index][0].score)

            handedness = "Unknown"
            if index < len(result.handedness) and result.handedness[index]:
                handedness = result.handedness[index][0].category_name

            hands.append({
                "points": points,
                "gesture": gesture,
                "confidence": score,
                "handedness": handedness,
            })
        return hands

    def close(self):
        self.recognizer.close()


class PersonSegmenter:
    """Segments the person in the webcam frame for the Thanos disappearance effect."""

    def __init__(self):
        selfie = getattr(mp, "solutions", None)
        if selfie is None or not hasattr(selfie, "selfie_segmentation"):
            raise RuntimeError(
                "This MediaPipe build does not expose selfie_segmentation. "
                "Install mediapipe==0.10.14 and restart the kernel."
            )
        self.segmenter = selfie.selfie_segmentation.SelfieSegmentation(
            model_selection=1
        )

    def mask(self, bgr_frame):
        rgb = cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB)
        result = self.segmenter.process(rgb)
        if result.segmentation_mask is None:
            return None
        return result.segmentation_mask

    def close(self):
        self.segmenter.close()


Overwriting src/hand_tracking.py


In [5]:
%%writefile src/vfx.py
import math
import random
from collections import deque

import cv2
import numpy as np

from config import Character


def palm_center(points):
    """Palm anchor from wrist and four MCP landmarks; fingers are not rendered."""
    return np.mean(points[[0, 5, 9, 13, 17], :2], axis=0)


def to_px(point, width, height):
    return (
        int(np.clip(point[0] * width, 0, width - 1)),
        int(np.clip(point[1] * height, 0, height - 1)),
    )


def glow(layer, center, radius, color, core=(245, 248, 255)):
    x, y = center
    radius = max(2, int(radius))
    cv2.circle(layer, (x, y), max(3, int(radius * 2.8)), color, -1, cv2.LINE_AA)
    cv2.circle(layer, (x, y), radius, core, -1, cv2.LINE_AA)


def bloom_composite(frame, overlay, blur=15, bloom_strength=0.9, detail_strength=0.95):
    soft = cv2.GaussianBlur(overlay, (0, 0), blur)
    frame[:] = cv2.addWeighted(frame, 1.0, soft, bloom_strength, 0)
    frame[:] = cv2.addWeighted(frame, 1.0, overlay, detail_strength, 0)


def ring(layer, center, radius, color, thickness=2, start=0, end=350):
    cv2.ellipse(layer, center, (max(2, int(radius)), max(2, int(radius))),
                0, start, end, color, thickness, cv2.LINE_AA)


def mandala(layer, center, radius, color, accent, t, dense=1.0):
    """Palm-centred sorcerer mandala. No landmarks or lines are drawn on fingers."""
    cx, cy = center
    radius = max(25, int(radius))
    scales = (1.0, .91, .79, .66, .53, .39, .25)
    for i, scale in enumerate(scales):
        r = int(radius * scale)
        start = int((t * (22 if i % 2 else -17) + i * 19) % 360)
        ring(layer, center, r, color if i % 2 == 0 else accent,
             3 if i < 2 else 1, start, start + 342)
    count = int(48 * dense)
    for i in range(count):
        a = math.tau * i / count + t * .035
        r = radius * (.82 if i % 4 else .72)
        x, y = int(cx + math.cos(a) * r), int(cy + math.sin(a) * r)
        length = 5 if i % 3 else 12
        tx, ty = -math.sin(a), math.cos(a)
        cv2.line(layer, (int(x-tx*length), int(y-ty*length)),
                 (int(x+tx*length), int(y+ty*length)),
                 accent if i % 4 == 0 else color, 1, cv2.LINE_AA)
    for sides, scale, rotation in ((8, .58, 22.5), (4, .40, 45), (6, .29, 0)):
        pts = []
        for i in range(sides):
            a = math.tau * i / sides + math.radians(rotation) + t * .025
            pts.append((int(cx + math.cos(a)*radius*scale),
                        int(cy + math.sin(a)*radius*scale)))
        cv2.polylines(layer, [np.asarray(pts, np.int32)], True, accent, 2, cv2.LINE_AA)
    cv2.circle(layer, center, max(5, int(radius*.12)), accent, 2, cv2.LINE_AA)
    cv2.circle(layer, center, max(3, int(radius*.055)), (255,255,255), -1, cv2.LINE_AA)


def particle_cloud(layer, center, radius, color, accent, t, count=100, seed=0):
    """Dense floating energy particles contained around the palm."""
    cx, cy = center
    rng = random.Random(int(t * 15) + seed)
    for i in range(count):
        a = rng.random() * math.tau + t * (0.35 + (i % 4) * .08)
        r = radius * (0.25 + 0.85 * rng.random())
        x = int(cx + math.cos(a) * r)
        y = int(cy + math.sin(a) * r * .78)
        cv2.circle(layer, (x, y), rng.choice((1, 1, 2, 3)),
                   accent if i % 5 == 0 else color, -1, cv2.LINE_AA)


def palm_arcs(layer, center, radius, color, accent, t, count=9):
    """Short broken arcs around the palm only (never connected to fingertips)."""
    for i in range(count):
        start = int((t * 80 + i * 360 / count) % 360)
        extent = 22 + (i % 3) * 12
        r = radius * (0.72 + .28 * ((i % 2)))
        cv2.ellipse(layer, center, (int(r), int(r*.78)), 0,
                    start, start + extent, color if i % 2 else accent,
                    2 if i % 3 else 3, cv2.LINE_AA)


class MagicRenderer:
    """Seven palm-centred procedural VFX styles with motion-adaptive scale.

    MediaPipe landmarks estimate palm location, hand size and movement.
    They are not drawn. These are real-time visual approximations, not movie footage.
    """
    def __init__(self):
        self.motion = {}

    def _motion_scale(self, points, width, height, hand_id):
        center = palm_center(points)
        center_px = np.array([center[0] * width, center[1] * height], dtype=np.float32)
        # Palm width gives a scale that adapts to the hand's distance from camera.
        palm_width = np.linalg.norm(points[5, :2] - points[17, :2]) * width
        palm_height = np.linalg.norm(points[0, :2] - points[9, :2]) * height
        hand_size = max(18.0, (palm_width + palm_height) * .5)
        old = self.motion.get(hand_id)
        if old is None:
            speed = 0.0
            smooth_speed = 0.0
        else:
            old_center, old_time, old_speed = old
            dt = max(1/120, min(.15, self._now - old_time))
            speed = float(np.linalg.norm(center_px - old_center) / dt)
            smooth_speed = .72 * old_speed + .28 * speed
        self.motion[hand_id] = (center_px, self._now, smooth_speed)
        # Motion increases the energy radius; hand size controls the base radius.
        movement_boost = np.clip(smooth_speed / 900.0, 0.0, .75)
        base = np.clip(hand_size * .92, 32.0, 105.0)
        return int(base * (1.0 + movement_boost)), float(movement_boost)

    def render_hand(self, frame, character: Character, hand, elapsed, hand_id=0):
        h, w = frame.shape[:2]
        points = hand["points"]
        palm = to_px(palm_center(points), w, h)
        self._now = elapsed
        radius, motion_boost = self._motion_scale(points, w, h, hand_id)
        # Keep the visual anchored at the palm. Finger tips are deliberately unused.
        overlay = np.zeros_like(frame)
        name = character.name
        pulse = 1.0 + .08 * math.sin(elapsed * 7.0)
        r = int(radius * pulse)

        if name == "DOCTOR STRANGE":
            mandala(overlay, palm, int(r * 1.30), (0, 155, 255), (80, 220, 255), elapsed, 1.25)
            palm_arcs(overlay, palm, r * .83, (0, 175, 255), (100, 235, 255), elapsed, 12)
            particle_cloud(overlay, palm, r*.85, (0, 150, 255), (130, 235, 255), elapsed, 65, 1)
            glow(overlay, palm, 9, (0, 175, 255), (255, 250, 225))

        elif name == "SCARLET WITCH":
            # Dense crimson chaos energy concentrated around the palm.
            glow(overlay, palm, int(r*.82), (0, 0, 205), (255, 245, 255))
            for i in range(7):
                a = elapsed * (1.8 if i % 2 else -1.4) + i * math.tau/7
                axes = (int(r*(.72+i*.045)), int(r*(.48+i*.035)))
                cv2.ellipse(overlay, palm, axes, math.degrees(a), 12, 338,
                            (0, 0, 190 + (i%2)*45), 3, cv2.LINE_AA)
            palm_arcs(overlay, palm, r*.95, (0,0,220), (95,70,255), elapsed, 14)
            particle_cloud(overlay, palm, r*1.15, (0,0,220), (100,80,255), elapsed, 145, 2)
            glow(overlay, palm, int(r*.22), (0,0,245), (255,255,255))

        elif name == "AGATHA HARKNESS":
            # Layered violet witchfire with dense lavender particles and smoky coils.
            glow(overlay, palm, int(r*.88), (175, 20, 205), (255, 235, 255))
            for i in range(9):
                a = elapsed * (1.2 if i%2 else -1.5) + i*20
                cv2.ellipse(overlay, palm, (int(r*(.62+i*.055)), int(r*(.42+i*.045))),
                            math.degrees(a), 0, 335, (195, 35+i%2*25, 235), 3, cv2.LINE_AA)
            palm_arcs(overlay, palm, r*.95, (185,25,220), (245,135,255), elapsed, 16)
            particle_cloud(overlay, palm, r*1.18, (185,25,220), (245,150,255), elapsed, 155, 3)
            glow(overlay, palm, int(r*.24), (205,45,240), (255,255,255))

        elif name == "LOKI":
            # Emerald illusion magic: elegant elliptical ribbons and green motes.
            glow(overlay, palm, int(r*.75), (25,145,20), (220,255,220))
            for i in range(8):
                angle = elapsed * (1.35 if i%2 else -1.1) + i*23
                cv2.ellipse(overlay, palm, (int(r*(.58+i*.06)), int(r*(.35+i*.045))),
                            math.degrees(angle), 25, 330,
                            (25+i*2, 145+(i%2)*65, 25), 3, cv2.LINE_AA)
            particle_cloud(overlay, palm, r*1.12, (30,165,25), (155,255,155), elapsed, 125, 4)
            palm_arcs(overlay, palm, r*.9, (30,180,35), (155,255,155), elapsed, 10)
            glow(overlay, palm, int(r*.20), (45,195,45), (255,255,255))

        elif name == "THOR":
            # Blue-white lightning arcs orbit and branch locally around the palm.
            glow(overlay, palm, int(r*.72), (255,150,30), (255,255,255))
            palm_arcs(overlay, palm, r*1.05, (255,205,65), (255,250,225), elapsed, 13)
            rng = random.Random(int(elapsed*28))
            for i in range(12):
                a = elapsed*.9 + i*math.tau/12
                reach = r*(.65 + .4*rng.random())
                end = (int(palm[0]+math.cos(a)*reach),
                       int(palm[1]+math.sin(a)*reach*.78))
                # Short local zig-zag arcs are confined to the palm aura.
                mid = ((palm[0]+end[0])//2 + rng.randint(-8,8),
                       (palm[1]+end[1])//2 + rng.randint(-8,8))
                cv2.polylines(overlay, [np.asarray([palm, mid, end], np.int32)],
                              False, (255,215,105), 2, cv2.LINE_AA)
                cv2.circle(overlay, end, 2, (255,255,255), -1, cv2.LINE_AA)
            particle_cloud(overlay, palm, r*1.1, (255,185,45), (255,255,255), elapsed, 100, 5)

        elif name == "WONG":
            mandala(overlay, palm, int(r*1.12), (0,135,245), (130,220,255), elapsed, 1.1)
            palm_arcs(overlay, palm, r*.9, (0,155,255), (190,235,255), elapsed, 10)
            particle_cloud(overlay, palm, r*.9, (0,135,245), (190,235,255), elapsed, 85, 6)

        elif name == "THE ANCIENT ONE":
            mandala(overlay, palm, int(r*1.38), (65,185,220), (190,245,255), elapsed, 1.5)
            palm_arcs(overlay, palm, r*1.05, (65,185,220), (220,250,255), elapsed, 16)
            particle_cloud(overlay, palm, r*1.12, (65,185,220), (220,250,255), elapsed, 120, 7)

        bloom_composite(frame, overlay, blur=max(10, int(r*.18)),
                        bloom_strength=.95 + motion_boost*.12, detail_strength=.92)

    def render_two_hand_field(self, frame, character, hands, elapsed):
        """No hand-to-hand connector: each palm gets its own independent VFX."""
        return


Overwriting src/vfx.py


In [ ]:
%%writefile src/main.py
import argparse
import time

import cv2

from config import CHARACTERS
from hand_tracking import HandTracker
from vfx import MagicRenderer


def draw_hud(frame, character, hands, fps, show_help=True):
    h, w = frame.shape[:2]

    panel = frame.copy()
    cv2.rectangle(panel, (0, 0), (w, 88), (12, 10, 24), -1)
    cv2.addWeighted(panel, 0.74, frame, 0.26, 0, frame)

    cv2.putText(
        frame, "MYSTIC MOTION", (18, 31),
        cv2.FONT_HERSHEY_DUPLEX, 0.78,
        (235, 235, 255), 2, cv2.LINE_AA
    )

    cv2.putText(
        frame, character.name, (18, 62),
        cv2.FONT_HERSHEY_SIMPLEX, 0.58,
        character.accent, 2, cv2.LINE_AA
    )

    cv2.putText(
        frame, f"HANDS: {len(hands)}   FPS: {fps:.0f}",
        (w - 230, 36),
        cv2.FONT_HERSHEY_SIMPLEX, 0.55,
        (235, 235, 245), 1, cv2.LINE_AA
    )

    if show_help:
        cv2.putText(
            frame,
            "1 Strange  2 Wanda  3 Agatha  4 Loki  5 Thor  6 Thanos",
            (16, h - 18),
            cv2.FONT_HERSHEY_SIMPLEX, 0.45,
            (210, 215, 235), 1, cv2.LINE_AA
        )

        cv2.putText(
            frame,
            "Q: Quit | H: Help | Magic activates automatically on detected palms",
            (16, h - 42),
            cv2.FONT_HERSHEY_SIMPLEX, 0.40,
            (180, 185, 205), 1, cv2.LINE_AA
        )


def run(camera_index=0, width=1280, height=720, mirror=True):
    cap = cv2.VideoCapture(camera_index)

    if not cap.isOpened():
        raise RuntimeError(
            f"Could not open camera index {camera_index}. "
            "Check macOS camera permission or try camera_index=1."
        )

    cap.set(cv2.CAP_PROP_FRAME_WIDTH, width)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, height)

    tracker = None

    try:
        tracker = HandTracker(max_hands=2)
        renderer = MagicRenderer()

        character_key = "1"
        show_help = True

        previous = time.perf_counter()
        fps = 0.0

        print("\nMYSTIC MOTION started.")
        print("1 Doctor Strange")
        print("2 Scarlet Witch")
        print("3 Agatha Harkness")
        print("4 Loki")
        print("5 Thor")
        print("6 Thanos")
        print("H = Help | Q = Quit")

        while True:
            ok, frame = cap.read()

            if not ok:
                print("Camera frame could not be read.")
                break

            if mirror:
                frame = cv2.flip(frame, 1)

            now = time.perf_counter()
            dt = max(now - previous, 1e-6)
            previous = now

            current_fps = 1.0 / dt
            fps = 0.9 * fps + 0.1 * current_fps if fps else current_fps

            hands = tracker.process(frame)
            character = CHARACTERS[character_key]

            # Render individual magic on every detected palm.
            for hand_id, hand in enumerate(hands):
                renderer.render_hand(
                    frame, character, hand, now, hand_id
                )

            # Render the third orb only when two palms face each other.
            if len(hands) >= 2:
                renderer.render_two_hand_field(
                    frame, character, hands, now
                )

            draw_hud(frame, character, hands, fps, show_help)

            cv2.imshow(
                "MYSTIC MOTION | Live Magic Experience",
                frame
            )

            key = cv2.waitKey(1) & 0xFF

            if key in (ord("q"), ord("Q"), 27):
                print("Closing Mystic Motion...")
                break

            elif key in (ord("h"), ord("H")):
                show_help = not show_help

            elif ord("1") <= key <= ord("6"):
                character_key = chr(key)
                print("Selected:", CHARACTERS[character_key].name)

    finally:
        if tracker is not None:
            tracker.close()

        cap.release()
        cv2.destroyAllWindows()


if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="Mystic Motion real-time palm VFX"
    )

    parser.add_argument("--camera", type=int, default=0)
    parser.add_argument("--width", type=int, default=1280)
    parser.add_argument("--height", type=int, default=720)
    parser.add_argument("--no-mirror", action="store_true")

    args = parser.parse_args()

    run(
        camera_index=args.camera,
        width=args.width,
        height=args.height,
        mirror=not args.no_mirror
    )

Overwriting src/main.py


In [ ]:
import ast
import importlib
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
SRC = PROJECT_ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

for filename in (
    "config.py",
    "hand_tracking.py",
    "vfx.py",
    "main.py"
):
    ast.parse((SRC / filename).read_text(encoding="utf-8"))
    print("Syntax OK:", filename)

import config
import hand_tracking
import vfx
import main

for module in (config, hand_tracking, vfx, main):
    importlib.reload(module)

expected = [
    "DOCTOR STRANGE",
    "SCARLET WITCH",
    "AGATHA HARKNESS",
    "LOKI",
    "THOR",
    "THANOS"
]

assert [c.name for c in config.CHARACTERS.values()] == expected
assert hasattr(hand_tracking, "HandTracker")
assert hasattr(vfx.MagicRenderer, "render_hand")
assert hasattr(vfx.MagicRenderer, "render_two_hand_field")
assert hasattr(vfx, "facing_each_other")

print("\nALL CHECKS PASSED")
print("Characters:", ", ".join(expected))

Syntax OK: config.py
Syntax OK: hand_tracking.py
Syntax OK: vfx.py
Syntax OK: main.py

ALL CHECKS PASSED
Characters: DOCTOR STRANGE, SCARLET WITCH, AGATHA HARKNESS, LOKI, THOR, WONG, THE ANCIENT ONE


In [8]:
from main import run

run(
    camera_index=0,
    width=1280,
    height=720,
    mirror=True,
)

I0000 00:00:1794744877.815080 2001101 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3
W0000 00:00:1794744877.855342 2001101 gesture_recognizer_graph.cc:129] Hand Gesture Recognizer contains CPU only ops. Sets HandGestureRecognizerGraph acceleration to Xnnpack.
I0000 00:00:1794744877.876258 2001101 hand_gesture_recognizer_graph.cc:250] Custom gesture classifier is not defined.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1794744877.925702 2005285 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794744877.962517 2005289 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794744877.965874 2005286 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback te

\nMYSTIC MOTION started.
Select characters using keys 1-7.
Magic is automatic whenever a hand is detected; no gesture is required.
Effects stay on the palm, contain no finger-connecting lines, and scale with movement.
H = Help | Q = Quit\n
Selected: SCARLET WITCH
Closing Mystic Motion...
